# Importing Necessary Libraries

In [ ]:
import os
import glob as glob
import matplotlib.pyplot as plt
import cv2
import requests
import random
import numpy as np

np.random.seed(42)

# Converting bounding boxes in YOLO format

In [ ]:
def yolo2bbox(bboxes):
    xmin, ymin = bboxes[0]-bboxes[2] / 2, bboxes[1] - bboxes[3] / 2
    xmax, ymax = bboxes[0]+bboxes[2] / 2, bboxes[1] + bboxes[3] / 2
    return xmin, ymin, xmax, ymax

In [ ]:
class_names = ['Ambulance', 'Bus', 'Car', 'Motorcycle', 'Truck']
colors = np.random.uniform(0, 255, size=(len(class_names), 3))

def plot_box(image, bboxes, labels):

    h, w, _ = image.shape
    for box_num, box in enumerate(bboxes):
        x1, y1, x2, y2 = yolo2bbox(box)

        # denormalize the coordinates
        xmin = int(x1 * w)
        ymin = int(y1 * h)
        xmax = int(x2 * w)
        ymax = int(y2 * h)
        width = xmax - xmin
        height = ymax - ymin

        class_name = class_names[int(labels[box_num])]

        cv2.rectangle(
            image,
            (xmin, ymin), (xmax, ymax),
            color=colors[class_names.index(class_name)],
            thickness=2
        )

        font_scale = min(1,max(3,int(w/500)))
        font_thickness = min(2, max(10,int(w/50)))

        p1, p2 = (int(xmin), int(ymin)), (int(xmax), int(ymax))

        # Text width and height
        tw, th = cv2.getTextSize(class_name, 0, fontScale=font_scale, thickness=font_thickness)[0]
        p2 = p1[0] + tw, p1[1] + -th - 10
        cv2.rectangle(
            image,
            p1, p2,
            color=colors[class_names.index(class_name)],
            thickness=-1,
        )
        cv2.putText(
            image,
            class_name,
            (xmin+1, ymin-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            font_thickness
        )
    return image

# Plotting images with the bounding boxes

In [ ]:
def plot(image_paths, label_paths, num_samples):
    all_training_images = glob.glob(image_paths)
    all_training_labels = glob.glob(label_paths)
    all_training_images.sort()
    all_training_labels.sort()

    num_images = len(all_training_images)

    plt.figure(figsize=(15, 12))
    for i in range(num_samples):
        j = random.randint(0,num_images-1)
        image = cv2.imread(all_training_images[j])
        with open(all_training_labels[j], 'r') as f:
            bboxes = []
            labels = []
            label_lines = f.readlines()
            for label_line in label_lines:
                label = label_line[0]
                bbox_string = label_line[2:]
                x_c, y_c, w, h = bbox_string.split(' ')
                x_c = float(x_c)
                y_c = float(y_c)
                w = float(w)
                h = float(h)
                bboxes.append([x_c, y_c, w, h])
                labels.append(label)
        result_image = plot_box(image, bboxes, labels)
        plt.subplot(2, 2, i+1)
        plt.imshow(result_image[:, :, ::-1])
        plt.axis('off')
    plt.subplots_adjust(wspace=0)
    plt.tight_layout()
    plt.show()

# Unzipping The dataset

In [ ]:
zip_path = '/content/archive_new.zip'
!unzip -q "$zip_path" -d /content/archive_new

# Creating Label and Image directory

In [ ]:
label_dir='/content/archive_new/Cars Detection/train/labels/*'
image_dir='/content/archive_new/Cars Detection/train/images/*'

# Visualising sample images

In [ ]:
plot(image_paths=image_dir,
    label_paths=label_dir,
    num_samples=4,
)

# Model YOLOV8

In [ ]:
!pip install ultralytics

In [ ]:
import os
from ultralytics import YOLO
# Load a model
model = YOLO("yolov8n.yaml")

In [ ]:
yaml_dir='/content/archive_new/Cars Detection/data.yaml'

In [ ]:
# Use the model
results = model.train(data=yaml_dir, epochs=100)

# Model Evaluation Section

In [ ]:
metrics = model.val()

# Predicting Images Section

In [ ]:
!yolo task=detect mode=predict model=/content/yolo11n.pt source='/content/archive_new/Cars Detection/test/images/59b1c028bd8b08dc_jpg.rf.hZvI4yexc1c6709HMKkq.jpg'

In [ ]:
img_path = '/content/archive_new/Cars Detection/test/images/59b1c028bd8b08dc_jpg.rf.hZvI4yexc1c6709HMKkq.jpg'
img = plt.imread(img_path)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis(False)
plt.show()